# Session 4: Machines that learn and see

So far, a vector has been an arrow: across, up, and sometimes forward. Today you'll see the other way to think about a vector, which is the way computers think about it: **a vector is a list of numbers**. Once you see it like that, vectors are everywhere.

By the end of this session you'll have:

1. Described colours and houses as vectors.
2. Priced a house by finding the most similar one, using nothing but subtraction and Pythagoras.
3. Edited photos and built a program that reads handwriting, using the same maths.
4. Watched a computer **learn** a rule by itself, by nudging numbers to make its mistakes smaller.
5. Raced your graphics card against your processor.
6. Done arithmetic with **words**: king − man + woman = ?

Every part uses only what you learned in sessions 1 to 3.

Run this cell first. It loads the tools for the rest of the notebook.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact

np.set_printoptions(suppress=True, precision=2)

## Part 1: A vector is a list of numbers

An arrow is one picture of a vector. But all a vector really is, is **a list of numbers in a fixed order**. The numbers don't have to mean "across" and "up". They can mean anything.

**A colour** on a screen is three numbers: how much **red**, **green** and **blue** light to mix, each from 0 to 255. So every colour is a 3D vector:

$$\text{orange} = \begin{pmatrix} 255 \\ 165 \\ 0 \end{pmatrix}$$

And because colours are vectors, you can do vector maths with them. The colour halfway between red and blue is the **midpoint** of their vectors: add them and halve the result.

In [ ]:
# ✏️ Change these: (red, green, blue), each from 0 to 255
colour_a = np.array([255, 0, 0])    # red
colour_b = np.array([0, 0, 255])    # blue

halfway = (colour_a + colour_b) / 2

fig, axes = plt.subplots(1, 3, figsize=(9, 2.6))
for ax, colour, name in zip(axes, [colour_a, halfway, colour_b], ["colour a", "halfway", "colour b"]):
    ax.add_patch(plt.Rectangle((0, 0), 1, 1, color=colour / 255))
    ax.set_title(f"{name}\n{tuple(int(round(c)) for c in colour)}")
    ax.axis("off")
plt.show()



**A house** can be a vector too. Describe it by 4 numbers:

$$\text{house} = \begin{pmatrix} \text{bedrooms} \\ \text{floor area (m}^2\text{)} \\ \text{garden (m}^2\text{)} \\ \text{distance to the station (km)} \end{pmatrix}$$

That's a vector in **4 dimensions**. You can't draw it, but you can still add, subtract and measure it, exactly like in 2D and 3D.

## Part 2: Pricing a house by its nearest neighbour

Here are 15 houses that have sold, with their prices in thousands of pounds. *(They're made up, but realistic.)*

Here's a simple way to guess the price of a new house: **find the sold house that is most similar, and guess the same price.** This is a real machine-learning method, called **nearest neighbour**.

"Most similar" means **the smallest distance between the two vectors**. You already know how to find that: subtract the vectors, then find the length of the answer. With 4 numbers, Pythagoras just gets 4 squares instead of 2:

$$\text{distance} = \sqrt{(\text{difference}_1)^2 + (\text{difference}_2)^2 + (\text{difference}_3)^2 + (\text{difference}_4)^2}$$

In [ ]:
# bedrooms, floor area (m²), garden (m²), km to station
houses = np.array([
    [1,  48,   0, 0.4], [2,  65,  20, 1.2], [2,  72,  60, 0.8], [3,  88,  40, 2.5],
    [3,  95, 120, 1.0], [3, 102,  80, 3.8], [4, 118, 150, 1.5], [4, 125,  90, 0.6],
    [4, 140, 200, 4.2], [5, 160, 180, 2.0], [5, 175, 250, 3.0], [2,  58,   0, 0.3],
    [3,  90,  10, 0.5], [1,  40,  15, 2.2], [4, 110,  60, 2.8],
])
prices = np.array([153, 205, 229, 237, 311, 277, 362, 383, 395, 474, 507, 194, 263, 114, 309])  # £ thousands

print(" house   bedrooms   area   garden   km     price")
for i, (h, p) in enumerate(zip(houses, prices)):
    print(f"{i:5}   {h[0]:8.0f}   {h[1]:4.0f}   {h[2]:6.0f}   {h[3]:3.1f}   £{p}k")

In [ ]:
def distance(a, b):
    # Subtract, then Pythagoras: square every number, add them up, square root
    return np.sqrt(np.sum((a - b) ** 2))


def nearest_house(new, data):
    distances = [distance(new, h) for h in data]
    best = int(np.argmin(distances))
    return best, distances


new_house = np.array([2, 90, 40, 1.0])

best, distances = nearest_house(new_house, houses)
print("The 3 closest houses:")
for i in np.argsort(distances)[:3]:
    print(f"  house {i:2}: {houses[i]}   distance {distances[i]:5.1f}   sold for £{prices[i]}k")
print(f"\nGuess: £{prices[best]}k (the same as house {best})")

Now experiment. Predict first each time.

1. Change `new_house` to a big house, like `[5, 170, 200, 2.5]`. Which house does it match?
2. **Something is wrong.** Set `new_house` back to `[2, 90, 40, 1.0]`, then change the bedrooms from `2` to `5`. Three extra bedrooms should make a big difference to the price. Does the guess change? Look at the numbers in each column and work out why.

### The problem: big numbers shout louder

Bedrooms go from 1 to 5, but floor area and garden size are in the **tens and hundreds**. When you subtract and square, a difference of 3 bedrooms only adds $3^2 = 9$, but a difference of 20 m² of garden adds $20^2 = 400$. So the distance is almost all floor area and garden, and the bedrooms barely count.

**The fix:** put every number on the same scale. Divide each column by its biggest value, so that every number ends up between 0 and 1. That's multiplying each number in the vector by its own scaling number, a close cousin of scalar multiplication.

Machine-learning engineers do this all the time. It's called **feature scaling**.

In [ ]:
biggest = houses.max(axis=0)       # the biggest value in each column
print("Dividing each column by:", biggest)

scaled_houses = houses / biggest

# ✏️ Try 2 bedrooms, then 5
new_house = np.array([5, 90, 40, 1.0])

best, distances = nearest_house(new_house / biggest, scaled_houses)
print(f"\nNearest house: {best} {houses[best]}")
print(f"Guess: £{prices[best]}k")

### Check yourself (no computer)

1. Find the distance between the houses $\begin{pmatrix} 2 \\ 60 \\ 10 \\ 1 \end{pmatrix}$ and $\begin{pmatrix} 4 \\ 63 \\ 10 \\ 1 \end{pmatrix}$.
2. A mansion with 9 bedrooms and 400 m² comes on the market. Why would nearest neighbour give a bad guess for it?

## Part 3: Machines that see

To a computer, a black-and-white photo is a grid of numbers: one number per pixel, from **0 (black) to 255 (white)**. Write all those numbers out in one long list and the whole picture is **one vector**, with hundreds of thousands of numbers in it.

Here's a photo, and a tiny square from it, zoomed in until you can read the numbers.

In [ ]:
from sklearn.datasets import load_sample_image

photo_colour = load_sample_image("china.jpg")       # a photo that comes with scikit-learn
photo = photo_colour.mean(axis=2)                   # average red, green and blue to get grey

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5))
ax1.imshow(photo, cmap="gray", vmin=0, vmax=255)
ax1.add_patch(plt.Rectangle((300, 180), 8, 8, fill=False, edgecolor="red", linewidth=2))
ax1.set_title("The photo (the red square is zoomed in on the right)")
ax1.axis("off")

patch = photo[180:188, 300:308]
ax2.imshow(patch, cmap="gray", vmin=0, vmax=255)
for r in range(8):
    for c in range(8):
        ax2.text(c, r, int(patch[r, c]), ha="center", va="center", fontsize=8,
                 color="white" if patch[r, c] < 128 else "black")
ax2.set_title("What the computer sees")
ax2.axis("off")
plt.show()

print(f"This photo is {photo.shape[0]} × {photo.shape[1]} pixels: one vector with {photo.size:,} numbers.")

Now everything you know about vectors becomes a way to edit photos:

| Vector maths | What it does to the photo |
|---|---|
| **Add** the same number to every pixel | Brighter (or darker, if it's negative) |
| **Multiply** every pixel by a number | More contrast (or less) |
| **255 − photo** | A negative: black becomes white |

Pixels can't go below 0 or above 255, so any number outside that range gets cut off. Use the sliders. Predict what each one does before you move it.

In [ ]:
@interact(brightness=(-150, 150, 10), contrast=(0.0, 3.0, 0.1), negative=False)
def edit_photo(brightness=0, contrast=1.0, negative=False):
    middle = 128
    edited = contrast * (photo - middle) + middle + brightness   # stretch around middle grey, then add
    if negative:
        edited = 255 - edited
    edited = np.clip(edited, 0, 255)                             # pixels must stay between 0 and 255
    plt.figure(figsize=(7, 4.5))
    plt.imshow(edited, cmap="gray", vmin=0, vmax=255)
    plt.axis("off")
    plt.show()

You can also blend two photos. If $\mathbf{a}$ and $\mathbf{b}$ are the two photo vectors, then

$$(1 - t)\,\mathbf{a} + t\,\mathbf{b}$$

is photo $\mathbf{a}$ when $t = 0$, photo $\mathbf{b}$ when $t = 1$, and a mix in between. This works in colour too, with 3 numbers per pixel. Mixing vectors like this is called a **linear combination**, and it's the topic of the second 3Blue1Brown video.

In [ ]:
flower_colour = load_sample_image("flower.jpg")   # the same size as the other photo


@interact(t=(0.0, 1.0, 0.05))
def blend(t=0.5):
    mixed = (1 - t) * photo_colour + t * flower_colour
    plt.figure(figsize=(7, 4.5))
    plt.imshow(mixed.astype(np.uint8))
    plt.axis("off")
    plt.show()

### A program that reads handwriting

Here's something more impressive. scikit-learn comes with **1,797 handwritten digits**, each scanned as a tiny 8 × 8 grid of pixels (0 is white, 16 is black). So each digit is a vector with 64 numbers.

To teach the computer to read, we don't write any rules about what a 7 looks like. We use the house trick again: **to read a new digit, find the stored digit with the smallest distance to it, and copy its answer.**

In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()
pictures, answers = digits.data, digits.target   # 64 numbers per digit, and what each digit really is

# Keep the first 1,500 as examples. Save the rest to test the program on digits it has never seen
known_pictures, known_answers = pictures[:1500], answers[:1500]
test_pictures, test_answers = pictures[1500:], answers[1500:]


def read_digit(picture):
    distances = np.sqrt(((known_pictures - picture) ** 2).sum(axis=1))   # distance to every known digit
    best = int(np.argmin(distances))
    return known_answers[best], best


def show(picture, title, ax):
    ax.imshow(picture.reshape(8, 8), cmap="gray_r", vmin=0, vmax=16)
    ax.set_title(title)
    ax.axis("off")


# ✏️ Change this to look at a different test digit (0 to 296)
which = 3

guess, best = read_digit(test_pictures[which])
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(6, 3))
show(test_pictures[which], f"New digit (really a {test_answers[which]})", ax1)
show(known_pictures[best], f"Closest known digit: a {guess}", ax2)
plt.show()

right = sum(read_digit(p)[0] == a for p, a in zip(test_pictures, test_answers))
print(f"On all {len(test_answers)} test digits, it got {right} right ({100 * right / len(test_answers):.0f}%).")

Here are the digits it got **wrong**. For each one, can you see why it made that mistake?

In [ ]:
mistakes = [i for i in range(len(test_answers)) if read_digit(test_pictures[i])[0] != test_answers[i]]

fig, axes = plt.subplots(2, 8, figsize=(12, 3.6))
for ax_new, ax_match, i in zip(axes[0], axes[1], mistakes[:8]):
    guess, best = read_digit(test_pictures[i])
    show(test_pictures[i], f"really {test_answers[i]}", ax_new)
    show(known_pictures[best], f"read as {guess}", ax_match)
plt.tight_layout()
plt.show()

Now **draw your own digit**. Use `#` for black, `+` for grey and `.` for white. Each row must be 8 characters, and there must be 8 rows.

Try these:

1. Draw a 1, a 0 and a 4. Does it read them correctly?
2. Find a drawing that fools it. What makes it go wrong?
3. This program never learned what a 7 *is*. How can it still read one?

## Part 4: Machines that learn

Nearest neighbour just **remembers** examples. Most machine learning works differently: the computer finds a set of numbers that capture a rule, and then throws the examples away.

Here's the simplest possible rule. Plot each house's price against its floor area, and draw a straight line:

$$\text{price} = \text{slope} \times \text{area} + \text{start}$$

Which line is best? We need a way to score a line. For each house, measure the **miss**: how far the line's price is from the real price. Square every miss (so misses above and below both count, and big misses count a lot), then take the average. That's the **error**. The best line is the one with the smallest error.

**Your turn first:** use the sliders to make the error as small as you can. Write down your best score.

In [ ]:
area, price = houses[:, 1], prices


def error_of(slope, start):
    misses = slope * area + start - price
    return np.mean(misses ** 2)


@interact(slope=(0.0, 5.0, 0.05), start=(-150.0, 250.0, 5.0))
def try_a_line(slope=1.0, start=50.0):
    xs = np.array([30, 190])
    plt.figure(figsize=(7, 4.5))
    plt.vlines(area, price, slope * area + start, color="tab:red", alpha=0.6, label="misses")
    plt.scatter(area, price, color="black", zorder=3, label="houses")
    plt.plot(xs, slope * xs + start, color="tab:blue", linewidth=2, label="your line")
    plt.xlabel("floor area (m²)")
    plt.ylabel("price (£ thousands)")
    plt.title(f"error = {error_of(slope, start):,.0f}")
    plt.legend(loc="upper left")
    plt.ylim(0, 650)
    plt.show()

### Now let the computer do it

The computer doesn't guess. It follows a simple loop:

1. Start with any line at all, even a terrible one.
2. Work out which way to nudge the slope and the start so that the error goes **down**.
3. Nudge them a tiny bit in that direction.
4. Go back to step 2, and repeat hundreds of times.

Each nudge makes the error a little smaller, so the line slides towards the best one. This is called **gradient descent**. Working out the direction in step 2 uses calculus, which you'll meet at A-level. For now, all that matters is: *nudge the numbers to make the mistakes smaller, and repeat.*

Press ▶ to watch it learn.

In [ ]:
# Behind the scenes, the areas are rescaled so the nudges work nicely (feature scaling again!)
mean_area, spread = area.mean(), area.std()
scaled_area = (area - mean_area) / spread

a, b = 0.0, 0.0              # start with a terrible line: everything costs £0
history = []
for step in range(201):
    slope, start = b / spread, a - b * mean_area / spread     # convert back to normal units
    history.append((slope, start, error_of(slope, start)))
    misses = a + b * scaled_area - price
    a -= 0.05 * 2 * np.mean(misses)                  # nudge each number a little,
    b -= 0.05 * 2 * np.mean(misses * scaled_area)    # in the direction that lowers the error

errors = [h[2] for h in history]


def show_step(step):
    slope, start, err = history[step]
    xs = np.array([30, 190])
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.3))
    ax1.vlines(area, price, slope * area + start, color="tab:red", alpha=0.6)
    ax1.scatter(area, price, color="black", zorder=3)
    ax1.plot(xs, slope * xs + start, color="tab:blue", linewidth=2)
    ax1.set(xlabel="floor area (m²)", ylabel="price (£ thousands)", ylim=(0, 650),
            title=f"step {step}: price = {slope:.2f} × area + {start:.0f}")
    ax2.plot(errors, color="gray")
    ax2.scatter([step], [err], color="tab:blue", zorder=3)
    ax2.set(xlabel="step", ylabel="error", yscale="log", title=f"error = {err:,.0f}")
    plt.show()


player = widgets.Play(min=0, max=200, step=2, interval=80)
slider = widgets.IntSlider(min=0, max=200, description="step")
widgets.jslink((player, "value"), (slider, "value"))
display(widgets.HBox([player, slider]), widgets.interactive_output(show_step, {"step": slider}))

Did you beat the computer? The computer's final line is the best possible straight line for these houses.

### The big reveal

This is how **all** modern AI is trained, including the word vectors you're about to use and models like ChatGPT. The only difference is size:

| | This line | Word vectors (Part 6) | A model like ChatGPT |
|---|---|---|---|
| Numbers being nudged | 2 | 40 million (100 for each of 400,000 words) | hundreds of billions |
| Examples it learned from | 15 houses | 6 billion words of Wikipedia and news | trillions of words |

Nobody typed in any of those numbers. They start out random and get nudged, over and over, until the mistakes are small. That's why people say AI models are **grown** more than they're programmed.

## Part 5: Why AI needs graphics cards

Every nudge is made of millions of tiny sums: multiply two numbers, add the results. Training means doing that billions of times.

- A computer's **processor (CPU)** has a few very clever cores. Yours probably has 6 to 16.
- A **graphics card (GPU)** has **thousands** of simpler cores. Your GTX 1660 Ti has **1,536**. It was built to work out the colour of millions of pixels at once, which is exactly the same kind of job: the same simple sum, done on lots of numbers at the same time.

It's like comparing 8 maths professors with 1,500 students who can each do one simple sum. For one hard problem, the professors win. For a million easy sums, the students win easily.

### Setting up your graphics card (one time only)

To use the GPU from Python you need the GPU version of **PyTorch**, a library for AI. In the terminal where you set up this project (with your environment active), run:

```
pip install torch --index-url https://download.pytorch.org/whl/cu130
```

It's a big download (a few GB). Then restart this notebook's kernel (**Kernel → Restart**) and run the setup cell at the top again.

In [ ]:
try:
    import torch
    if torch.cuda.is_available():
        print("PyTorch can see your graphics card:", torch.cuda.get_device_name(0))
    else:
        print("PyTorch is installed, but it can't see a graphics card. The race below will only use the CPU.")
except ImportError:
    torch = None
    print("PyTorch isn't installed yet. See the instructions above. The race below will only use the CPU.")

### The race

Here's a job just like the ones in Part 6: we have **50,000 vectors** with **100 numbers** each, and for each of **1,000** new vectors we want to find the closest one. That's 50 million distances.

Both the CPU and the GPU do exactly the same sums. Run it and see who wins.

In [ ]:
import time

rng = np.random.default_rng(0)
library = rng.standard_normal((50_000, 100)).astype(np.float32)
queries = rng.standard_normal((1_000, 100)).astype(np.float32)


def closest_cpu(queries, library):
    # Squared distances for every pair at once (a shortcut that gives the same answer as subtracting)
    d = (queries ** 2).sum(1)[:, None] + (library ** 2).sum(1)[None, :] - 2 * queries @ library.T
    return d.argmin(axis=1)


start = time.perf_counter()
answer_cpu = closest_cpu(queries, library)
cpu_time = time.perf_counter() - start
print(f"CPU: {cpu_time:.3f} seconds")

if torch is not None and torch.cuda.is_available():
    q, lib = torch.from_numpy(queries).cuda(), torch.from_numpy(library).cuda()

    def closest_gpu(q, lib):
        d = (q ** 2).sum(1)[:, None] + (lib ** 2).sum(1)[None, :] - 2 * q @ lib.T
        return d.argmin(dim=1)

    closest_gpu(q, lib)              # a warm-up run: the first go is always slow
    torch.cuda.synchronize()
    start = time.perf_counter()
    answer_gpu = closest_gpu(q, lib)
    torch.cuda.synchronize()         # wait for the GPU to really finish before stopping the clock
    gpu_time = time.perf_counter() - start
    print(f"GPU: {gpu_time:.3f} seconds")
    print(f"The GPU was {cpu_time / gpu_time:.0f} times faster, and the answers agree:",
          bool((answer_gpu.cpu().numpy() == answer_cpu).mean() > 0.99))
else:
    print("No GPU available, so there's nothing to race against.")

Try making the job **smaller**, for example 100 vectors in the library instead of 50,000. Does the GPU still win? *(Sending numbers to the graphics card takes time, so for small jobs it isn't worth it.)*

## Part 6: Words as vectors

Researchers at Stanford trained a model called **GloVe** on 6 billion words of Wikipedia and news, using the nudge-and-repeat loop from Part 4. The result: every word gets a vector of **100 numbers**. The rule it learned is roughly: *words that appear in similar sentences should get similar vectors.*

This folder contains the vectors for the 50,000 most common words. Here's what one looks like:

In [ ]:
glove = np.load("data/glove-50k-100d.npz")
words = list(glove["words"])
word_vectors = glove["vectors"].astype(np.float32)
row = {w: i for i, w in enumerate(words)}   # look up which row a word is in


def vec(word):
    return word_vectors[row[word]]


print(f"{len(words):,} words, each with {word_vectors.shape[1]} numbers.\n")
print("The first 10 numbers for 'king':", vec("king")[:10])

On their own, those numbers mean nothing. No single number means "royal" or "male". The meaning is in how the vectors sit **compared to each other**.

So let's use the same algorithm a third time. Houses, digits, and now words: **find the closest vectors.**

In [ ]:
def closest_words(target, n=10, skip=()):
    distances = np.sqrt(((word_vectors - target) ** 2).sum(axis=1))
    order = np.argsort(distances)
    return [(words[i], distances[i]) for i in order if words[i] not in skip][:n]


# ✏️ Try any (lower-case) word
word = "football"

for w, d in closest_words(vec(word), skip=[word]):
    print(f"{d:5.2f}  {w}")

Try some of your own: a game, a food, a country, a footballer's surname. Do the closest words make sense? Find one that surprises you.

### king − man + woman = ?

Here's the famous one. In session 1 you learned that **equal vectors** are the same movement, wherever they start. If the "man → king" movement were the same as the "woman → queen" movement, then

$$\text{king} - \text{man} = \text{queen} - \text{woman}$$

which rearranges to

$$\text{king} - \text{man} + \text{woman} = \text{queen}$$

The four words would make a **parallelogram**.

We can't draw 100 dimensions, but we can draw a **shadow** of them in 2D, like the shadow of a 3D object on a wall. Shadows squash things, so it won't be perfect.

In [ ]:
pairs = [("man", "woman"), ("king", "queen"), ("boy", "girl"), ("prince", "princess"), ("uncle", "aunt")]
chosen = [w for pair in pairs for w in pair]

# Find the 2D "shadow" that keeps these words as spread out as possible
points = np.array([vec(w) for w in chosen])
centre = points.mean(axis=0)
_, _, directions = np.linalg.svd(points - centre)
shadow = (points - centre) @ directions[:2].T
where = dict(zip(chosen, shadow))

plt.figure(figsize=(7, 6))
for male, female in pairs:
    p, q = where[male], where[female]
    plt.annotate("", xy=q, xytext=p, arrowprops=dict(arrowstyle="->", color="tab:purple", lw=2))
for w, (x, y) in where.items():
    plt.scatter(x, y, color="black")
    plt.annotate(w, (x, y), textcoords="offset points", xytext=(6, 6), fontsize=12)
plt.title("A 2D shadow of the 100D word vectors\n(each arrow goes from the male word to the female word)")
plt.axis("equal")
plt.show()

The arrows aren't perfectly equal, because this is only a shadow, but they point in roughly the same direction. Now let's do the real calculation, in all 100 dimensions.

One detail: the closest vector to $\text{king} - \text{man} + \text{woman}$ is often **king** itself, because one small step from king is still near king. So, like everyone who does this, we leave out the three words we started with.

In [ ]:
def analogy(a, b, c, n=5):
    # a is to b as c is to ...?
    target = vec(b) - vec(a) + vec(c)
    print(f"{a} → {b}, so {c} → ?")
    for w, d in closest_words(target, n=n, skip=[a, b, c]):
        print(f"   {d:5.2f}  {w}")
    print()


analogy("man", "king", "woman")
analogy("france", "paris", "japan")
analogy("big", "bigger", "small")
analogy("walk", "walked", "swim")

### Your turn

Write your own analogies with `analogy("...", "...", "...")`. Some ideas:

- countries and their languages (`"germany", "german", "italy"`)
- a footballer and their club, or a club and its city
- a word and its opposite

Some will work beautifully and some will fail. When one fails, think about why: maybe the words don't appear in sentences together often enough.

**Something to think about:** these vectors were learned from human writing, so they also learn human **biases**. Try `analogy("man", "doctor", "woman")` and `analogy("man", "programmer", "woman")`. If the answers look unfair, it's because the 6 billion words they learned from were written that way. AI companies spend a lot of effort finding and fixing this.

### Check yourself (no computer)

1. Why do we leave out the three starting words when we look for the answer?
2. In words, what does the vector $\text{king} - \text{man}$ represent?
3. Nearest neighbour priced houses, read handwriting and found similar words. What was the same each time, and what was different?

## What you've seen today

- A vector is just a **list of numbers**, and the numbers can mean anything: colours, houses, pixels, words.
- **One algorithm**, "find the nearest vector", priced houses, read handwriting and found related words. It only needs subtraction and Pythagoras.
- Computers **learn** by nudging numbers to make their mistakes smaller, over and over. Every modern AI is trained that way.
- **GPUs** make it fast by doing thousands of simple sums at the same time.